## Config

In [1]:
import os, json, re, math
from time import time
from pathlib import Path
from tqdm import tqdm

import torch
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"*** Using device: {device.type} ***")

colab = False

if colab:
  from google.colab import drive
  drive.mount('/content/drive')
  ROOT_PATH = Path("/content/drive/MyDrive/ML-Projects/CausalLSTM")
else:
  ROOT_PATH = Path.cwd()

paths = {
        "data": ROOT_PATH / "data/",
        "config": ROOT_PATH / "config",
        "models": ROOT_PATH / "models",
    }

for key, path in paths.items():
        path.mkdir(parents=True, exist_ok=True)

SEED = 42

*** Using device: cuda ***


In [14]:
### WRITE CONFIG FILE ###

config = {
    "vocab_size": 25_000,
    "seq_len": 32,
    "batch_size": 64,
    "n_epochs": 60,
    "enable_mixed_precision": True if device.type == "cuda" else False,
    "grad_clip_norm": 0.5,
    "early_stopping_patience": 3,
    "early_stopping_epsilon": 3e-4,
    "model_params": {
        "embedding_dim": 512,
        "hidden_dim": 768,
        "num_layers": 2,
        "lstm_dropout_p": 0.25,
        "emb_dropout_p": 0.2,
        "out_dropout_p": 0.5,
    },
    "optimizer_params": {
        "lr": 2e-3,
        "weight_decay": 2e-5,
    },
    "lr_scheduler_params": {
        "factor": 0.5,
        "patience": 2,
        "threshold": 2e-3
    }
}

with open(paths["config"] / "config.json", "w", encoding="utf-8") as f:
    json.dump(config, f)
print(f"*** config saved to {paths["config"]} ***")

*** config saved to /mnt/c/Users/ASUS/Documents/Machine-Learning/Projects/CausalLSTM/config ***


## Prepare Data

In [15]:
data_dir = "data/clean_conan_doyle/"
corpus = ""
for f in os.listdir(data_dir):
    if f.endswith(".txt"):
        with open(os.path.join(data_dir, f), "r", encoding="utf-8") as c:
            corpus += (c.read() + " <eos> ")

print(f"*** count of chars: {len(corpus):,} | count of splits: {len(corpus.split()):,} ***")
print(f"*** count of unique splits lowercased {len(set(corpus.lower().split())):,}")

*** count of chars: 2,156,155 | count of splits: 370,831 ***
*** count of unique splits lowercased 29,738


In [16]:
from tokenizer import Tokenizer
from utils import normalize_text
from dataset import TruncatedBPTTDataset
    
tokenizer = Tokenizer(tokenize_method="nltk", preprocessor=normalize_text)
tokenizer.build_vocab([corpus])
tokenizer.save(paths["models"] / "tokenizer.json")
corpus_ids = tokenizer.encode(corpus)

train_ds = TruncatedBPTTDataset(corpus_ids, config["batch_size"], config["seq_len"])
train_ds.get_info()

*** 16,233 tokens added to vocab ***
*** tokenizer saved to /mnt/c/Users/ASUS/Documents/Machine-Learning/Projects/CausalLSTM/models/tokenizer.json ***
*** batch size: 64 | sequence len: 32 ***
*** stream lenght: 6790 | number of batches: 212 ***


## Training

In [17]:
from model import CausalLSTM, detach_hidden

torch.manual_seed(SEED)
model = CausalLSTM(tokenizer.get_vocab_size(), **config["model_params"]).to(device)
print(f"*** total count of trainable parameters: {sum([p.numel() for p in model.parameters() if p.requires_grad]):,} ***")
print(model)

loss_fn = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.NAdam(model.parameters(), **config["optimizer_params"])
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, **config["lr_scheduler_params"])

*** total count of trainable parameters: 17,384,297 ***
CausalLSTM(
  (embedding): Embedding(16233, 512)
  (emb_dropout): Dropout(p=0.2, inplace=False)
  (lstm): LSTM(512, 768, num_layers=2, batch_first=True, dropout=0.25)
  (out_dropout): Dropout(p=0.5, inplace=False)
  (proj): Linear(in_features=768, out_features=512, bias=True)
  (fc): Linear(in_features=512, out_features=16233, bias=True)
)


In [18]:
@torch.no_grad()
def evaluate(eval_dataset, disable_progress_bar=True):
    model.eval()
    total_loss = 0.0
    hidden = model.init_hidden(config["batch_size"])
    for idx in tqdm(range(len(eval_dataset)), disable = disable_progress_bar):
        X, Y = eval_dataset[idx]
        X, Y = X.to(device), Y.to(device)
        hidden = detach_hidden(hidden)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=config["enable_mixed_precision"]):
            logits, hidden = model(X, hidden)
            total_loss += loss_fn(logits, Y).item()
    return total_loss / len(eval_dataset)

In [19]:
def train(saved_checkpoint_path = None):
    if saved_checkpoint_path is not None:
        model.load_state_dict(
            torch.load(saved_checkpoint_path, map_location=device, weights_only=True)
            )
    train_logs = {"train_loss":[] , "val_loss":[] , "val_metric":[], "lr":[]}
    model.train()
    scaler = torch.amp.GradScaler(enabled = config["enable_mixed_precision"])
    best_loss, es_counter = float('inf'), 0

    for epoch in range(config["n_epochs"]):
        start_time = time()
        model.train()
        total_loss = 0.0
        hidden = model.init_hidden(config["batch_size"])

        for idx in tqdm(range(len(train_ds)), desc=f"Epoch {epoch+1}/{config["n_epochs"]}"):
            X, Y = train_ds[idx]
            X, Y = X.to(device), Y.to(device)
            optimizer.zero_grad(set_to_none=True)
            # detach hidden from current graph
            hidden = detach_hidden(hidden)
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled = config["enable_mixed_precision"]):
                logits, hidden = model(X, hidden)
                loss = loss_fn(logits, Y)
            total_loss += loss.item()
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            # clip gradients to avoid exloding
            nn.utils.clip_grad_norm_(model.parameters(), config["grad_clip_norm"])
            scaler.step(optimizer)
            scaler.update()

        # logger
        train_logs["train_loss"].append(total_loss / len(train_ds))
        #val_loss = evaluate(val_ds)
        #train_logs["val_loss"].append(val_loss)
        #train_logs["val_metric"].append(math.exp(val_loss))
        train_logs["lr"].append(optimizer.param_groups[0]['lr'])

        print(f"\r Epoch {epoch + 1}/{config["n_epochs"]}", end="")
        print(f", train loss: {train_logs["train_loss"][-1]:.4f}", end="")
        print(f', train perplexity: {math.exp(train_logs["train_loss"][-1]):.4f}', end="")
        #print(f', val loss: {train_logs["val_loss"][-1]:.4f}', end="")
        #print(f', val perplexity: {train_logs["val_metric"][-1]:.4f}', end="")
        print(f", lr: {train_logs["lr"][-1]}", end="")
        print(f', epoch time: {time() - start_time:.2f}s')

        # learning rate scheduler
        #scheduler.step(val_loss)
        torch.save(model.state_dict(), paths["models"] / f"CausualLSTM_ckpnt_{epoch+1}.pt")

        # early stopping
        """
        diff = best_loss - val_loss
        if diff >= config["early_stopping_epsilon"]:
            best_loss = val_loss
            es_counter = 0
        else:
            es_counter += 1
        if es_counter >= config["early_stopping_patience"]:
            print(f"*** early stopping triggered at epoch: {epoch+1} ***")
            break
        """
    return model, train_logs

In [20]:
ckpnt = str(paths["models"] / f"CausualLSTM_ckpnt_{30}.pt")
model, train_logs = train()

with open(paths["models"] / "train_logs.json", "w") as f:
    json.dump(train_logs, f)

Epoch 1/60: 100%|█████████████████████████████████████████████████████████████████████| 212/212 [00:12<00:00, 17.12it/s]


 Epoch 1/60, train loss: 5.8985, train perplexity: 364.4913, lr: 0.002, epoch time: 12.39s


Epoch 2/60: 100%|█████████████████████████████████████████████████████████████████████| 212/212 [00:09<00:00, 21.90it/s]


 Epoch 2/60, train loss: 4.9464, train perplexity: 140.6728, lr: 0.002, epoch time: 9.68s


Epoch 3/60: 100%|█████████████████████████████████████████████████████████████████████| 212/212 [00:12<00:00, 17.59it/s]


...

 Epoch 58/60, train loss: 3.0242, train perplexity: 20.5767, lr: 0.002, epoch time: 11.99s


Epoch 59/60: 100%|████████████████████████████████████████████████████████████████████| 212/212 [00:09<00:00, 22.98it/s]


 Epoch 59/60, train loss: 3.0243, train perplexity: 20.5797, lr: 0.002, epoch time: 9.23s


Epoch 60/60: 100%|████████████████████████████████████████████████████████████████████| 212/212 [00:11<00:00, 17.70it/s]


 Epoch 60/60, train loss: 3.0189, train perplexity: 20.4695, lr: 0.002, epoch time: 11.98s


## Inference

In [9]:
import json, torch
from model import CausalLSTM
from tokenizer import Tokenizer

checkpoint_path = "models/CausualLSTM_ckpnt_60.pt"
config_path = "config/config.json"
tokenizer_path = "models/tokenizer.json"

with open(config_path, "r") as f:
    config = json.load(f)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = Tokenizer()
tokenizer.load_from_file(tokenizer_path)
model = CausalLSTM(tokenizer.get_vocab_size(), **config["model_params"]).to(device)

model.load_state_dict(torch.load(checkpoint_path, map_location=device, weights_only=True))

<All keys matched successfully>

In [10]:
def generate(model, tokenizer, config, device, init_word, max_new_tokens=50, temperature=1.0, seed=42):
    torch.manual_seed(seed)
    model.eval()
    hidden = model.init_hidden(1)
    tkns = ["<unk>", "<eos>", init_word]
    unk_id, eos_id, init_idx = [tokenizer.token_to_id(t) for t in tkns]
    if init_idx is not None:
        input_ = torch.tensor(init_idx, dtype=torch.long).reshape(1,-1).to(device)
    else:
        raise Exception(f"{init_word} is not in vocab")
        
    generated_words = [init_word]
    with torch.no_grad():
        for _ in range(max_new_tokens):
            output, hidden = model(input_, hidden)
            probs = output.squeeze().div(temperature).exp().cpu()
            while True:
                token_idx = torch.multinomial(probs, 1)[0]
                if token_idx != unk_id: break
            input_.fill_(token_idx)
            generated_words.append(tokenizer.id_to_token(token_idx) if token_idx != eos_id else "\n")
    return " ".join(generated_words)

In [11]:
for t in [0.5, 0.7, 0.9, 1.1, 1.5, 2.0]:
    print("="*30)
    print(generate(model, tokenizer, config, device, init_word = "sherlock", max_new_tokens=50, temperature=t, seed=45))

sherlock holmes , he said , and i have only been in the evening before the day i came down . 
 i had already been able to look into it , for the whole thing was a strange and inexplicable one . 
 it was not for the time that
sherlock holmes stuck his own methods in afghanistan , on which the other lived with him , it was a devil , were it only to be the truth which he could tell . all that i was engaged to him , but i thought that he had better done it
sherlock holmes stuck his own compliment in inference . 
 for these circumstances we found that the crime was drawn up , since the second were a relation of the crime , and the point has been important . three times later six and eight years ago , when holmes was
sherlock holmes stuck his own compliment or deeper than his diseased thought . we found that we were best briony lodge , while i led to a secluded road fringed with fresh houses , each too sufferer on them . there were no furniture save a small lawn , with no
sherlock holmes stu

In [13]:
print(generate(model, tokenizer, config, device, init_word = "sherlock", max_new_tokens=50, temperature=0.9, seed=44))

sherlock holmes was a powerful sleeper . he was deeply interested by the bearing of the favourite and napoleon , but he carried it out with his own head , and a bed which struck me the food and apparently brought me into a room upon his rightly . you were
